In [1]:
%pip install onnx onnxruntime

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.5 MB ? eta -:--:--
   -------------- ------------------------- 4.7/13.5 MB 25.9 MB/s eta 0:00:01
   -------------------------------- ------- 11.0/13.5 MB 28.7 MB/s eta 0:00:01
   ---------------------------------------- 13.5/13.5 MB 24.8 MB/s  0:00:00

   ---------------------------------------- 0/5 [flatbuffers]
   -------- ------------------------------- 1/5 [pyreadline3]
   -------- ------------------------------- 1/5 [pyreadline3]
   -------- ------------------------------- 1/5 [pyreadline3]
   -------- ------------------------------- 1/5 [pyreadline3]
   -------- ------------------------------- 1/5 [pyreadline3]
   ---------------- ----------------------- 2/5 [humanfriendly]
   ---------------- ----------------------- 2/5 [humanfriendly]
   ---------------- ----------------------- 2/5 [humanfriendly]
   ------------------------ --------------- 3/5 [color

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import torch
import torch.nn as nn
import onnx

# 1. REDEFINE THE ARCHITECTURE (Required for the new file)
class BatterySoCEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output_layer = nn.Linear(8, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)

# Instantiate the model (this simulates the one we trained yesterday)
model = BatterySoCEstimator()

# CRITICAL STEP: Put the model into "Evaluation Mode"
# This turns off training-specific features and locks the weights for production
model.eval() 

# 2. CREATE THE TRACER SIGNAL
# We create a random 1x3 tensor (Voltage, Current, Temp) just so ONNX can track the math
tracer_signal = torch.randn(1, 3)

# 3. EXPORT TO ONNX
onnx_filename = "battery_soc_model.onnx"

torch.onnx.export(
    model,                      # The PyTorch model
    tracer_signal,              # The signal used to trace the path
    onnx_filename,              # The output file name
    export_params=True,         # Save the trained weights inside the file
    opset_version=11,           # The ONNX standard version
    input_names=['sensor_data'],   # Labeling the input pin for the C++ engineers
    output_names=['soc_estimate']  # Labeling the output pin
)

print(f"✅ Production Build Complete! Model exported to: {onnx_filename}")

W0226 13:29:57.405000 30640 site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BatterySoCEstimator([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BatterySoCEstimator([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


d:\Software\Anaconda\envs\ai_lab\lib\copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\vijay\AppData\Roaming\Python\Python310\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "C:\Users\vijay\AppData\Roaming\Python\Python310\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\vijay\AppData\Roaming\Python\Python310\site-packages\onnxscript\version_converter\__init__.py", line 

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
✅ Production Build Complete! Model exported to: battery_soc_model.onnx


In [3]:
import onnxruntime as ort
import numpy as np
import time

# 1. INITIALIZE THE ONNX ENGINE
# We load the frozen C++ model file we just created
ort_session = ort.InferenceSession("battery_soc_model.onnx")

# 2. PREPARE THE PRODUCTION INPUT
# We use our standard test values: 390.0V, -50.0A, 25.0°C.
# Notice we explicitly force it to be a 32-bit float (np.float32). 
# Embedded hardware is strictly typed and will crash if you pass it a 64-bit float by accident!
dummy_input = np.array([[390.0, -50.0, 25.0]], dtype=np.float32)

print("⚡ Firing up Edge Inference...")

# 3. RUN THE INFERENCE (And time it)
start_time = time.perf_counter()

# We tell the engine to run, specifying the exact input/output pin names we mapped earlier
outputs = ort_session.run(
    ['soc_estimate'],             # The output pin we want to read
    {'sensor_data': dummy_input}  # The input pin we are feeding
)

end_time = time.perf_counter()

# 4. PARSE THE RESULTS
# ONNX returns a list containing our NumPy array, so we just extract the raw number
soc_prediction = outputs[0][0][0]
inference_time_ms = (end_time - start_time) * 1000

print("-" * 40)
print(f"🔋 Predicted State of Charge: {soc_prediction:.2f}%")
print(f"⏱️ Execution Time: {inference_time_ms:.4f} milliseconds")
print("-" * 40)

⚡ Firing up Edge Inference...
----------------------------------------
🔋 Predicted State of Charge: 0.46%
⏱️ Execution Time: 0.8280 milliseconds
----------------------------------------
